# 02 — Curva ABC de Estoque
**Dataset:** Inventory Analysis Case Study  
**Objetivo:** Classificar os produtos pela Curva ABC com base no valor de vendas. Identificar os 20% de produtos que geram 80% da receita e os que drenam capital sem retorno.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

sales = pd.read_csv('../data/raw/SalesFINAL12312016.csv')
beg   = pd.read_csv('../data/raw/BegInvFINAL12312016.csv')

print(f'Vendas: {sales.shape[0]:,} registros')
print(f'Produtos unicos nas vendas: {sales["Brand"].nunique():,}')
sales.head(3)

Vendas: 422,077 registros
Produtos unicos nas vendas: 6,396


,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,1/1/2016,750.0,1.0,0.79,12546.0,JIM BEAM BRANDS COMPANY
1,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,2,32.98,16.49,1/2/2016,750.0,1.0,1.57,12546.0,JIM BEAM BRANDS COMPANY
2,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,1/3/2016,750.0,1.0,0.79,12546.0,JIM BEAM BRANDS COMPANY


## 1. Receita Total por Produto

In [2]:
receita = sales.groupby(['Brand', 'Description']).agg(
    Receita=('SalesDollars', 'sum'),
    Quantidade=('SalesQuantity', 'sum'),
    Transacoes=('SalesQuantity', 'count')
).reset_index().sort_values('Receita', ascending=False)

print(f'Total de produtos vendidos: {len(receita):,}')
print(f'Receita total: R$ {receita["Receita"].sum():,.2f}')
receita.head(10)

Total de produtos vendidos: 6,396
Receita total: R$ 12,639,812.24


,Brand,Description,Receita,Quantidade,Transacoes
1470,4261,Capt Morgan Spiced Rum,178360.89,8111,935
1147,3545,Ketel One Vodka,144012.17,4783,786
364,1233,Jack Daniels No 7 Black,132047.31,3669,803
2271,8068,Absolut 80 Proof,112267.62,4338,822
1083,3405,Tito's Handmade Vodka,103105.62,3438,764
1312,3858,Grey Goose Vodka,91449.88,3812,783
1452,4227,Bacardi Superior Rum Trav,74802.42,4158,773
714,2589,Jameson Irish Whiskey,73056.35,1765,600
711,2585,Jameson Irish Whiskey,69492.78,2922,779
407,1376,Jim Beam,68580.02,2998,838


## 2. Classificacao ABC
> **A:** 80% da receita acumulada  
> **B:** 80% a 95% da receita acumulada  
> **C:** 95% a 100% da receita acumulada

In [3]:
receita['Receita_Pct'] = receita['Receita'] / receita['Receita'].sum() * 100
receita['Receita_Acum'] = receita['Receita_Pct'].cumsum()

def classificar_abc(pct_acum):
    if pct_acum <= 80:
        return 'A'
    elif pct_acum <= 95:
        return 'B'
    else:
        return 'C'

receita['Classe'] = receita['Receita_Acum'].apply(classificar_abc)

resumo_abc = receita.groupby('Classe').agg(
    Produtos=('Brand', 'count'),
    Receita=('Receita', 'sum')
).reset_index()
resumo_abc['Pct Produtos (%)'] = (resumo_abc['Produtos'] / receita.shape[0] * 100).round(1)
resumo_abc['Pct Receita (%)'] = (resumo_abc['Receita'] / receita['Receita'].sum() * 100).round(1)

print('=== RESUMO CURVA ABC ===')
print(resumo_abc[['Classe', 'Produtos', 'Pct Produtos (%)', 'Pct Receita (%)']].to_string(index=False))

=== RESUMO CURVA ABC ===
Classe  Produtos  Pct Produtos (%)  Pct Receita (%)
     A      1359              21.2             80.0
     B      1629              25.5             15.0
     C      3408              53.3              5.0


## 3. Grafico de Pareto (Curva ABC)

In [4]:
receita_plot = receita.reset_index(drop=True)
receita_plot['Ranking'] = range(1, len(receita_plot) + 1)
receita_plot['Pct Produtos'] = receita_plot['Ranking'] / len(receita_plot) * 100

fig = go.Figure()

# Barras de receita coloridas por classe
cores_abc = {'A': '#2ecc71', 'B': '#f39c12', 'C': '#e74c3c'}
for classe in ['A', 'B', 'C']:
    mask = receita_plot['Classe'] == classe
    fig.add_trace(go.Bar(
        x=receita_plot[mask]['Pct Produtos'],
        y=receita_plot[mask]['Receita_Pct'],
        name=f'Classe {classe}',
        marker_color=cores_abc[classe],
        opacity=0.8
    ))

# Linha acumulada
fig.add_trace(go.Scatter(
    x=receita_plot['Pct Produtos'],
    y=receita_plot['Receita_Acum'],
    name='Receita Acumulada (%)',
    yaxis='y2',
    line=dict(color='#2c3e50', width=2)
))

fig.add_hline(y=80, line_dash='dash', line_color='#2ecc71', annotation_text='80% (A)', yref='y2')
fig.add_hline(y=95, line_dash='dash', line_color='#f39c12', annotation_text='95% (B)', yref='y2')

fig.update_layout(
    title='Curva ABC de Estoque por Receita de Vendas',
    xaxis_title='% de Produtos (por ranking)',
    yaxis_title='% da Receita Individual',
    yaxis2=dict(title='% Receita Acumulada', overlaying='y', side='right', range=[0, 105]),
    height=500,
    barmode='stack',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

## 4. Top 20 Produtos Classe A

In [5]:
classe_a = receita[receita['Classe'] == 'A'].head(20).copy()

fig = px.bar(
    classe_a.sort_values('Receita'),
    x='Receita', y='Description',
    orientation='h',
    text='Receita',
    color='Receita',
    color_continuous_scale='Greens',
    title='Top 20 Produtos Classe A (maiores geradores de receita)',
    labels={'Description': 'Produto', 'Receita': 'Receita (R$)'}
)
fig.update_traces(texttemplate='R$ %{text:,.0f}', textposition='outside')
fig.update_layout(height=580, coloraxis_showscale=False)
fig.show()

## 5. Produtos Classe C com Alto Estoque (Capital Parado)

In [6]:
# Cruzar classificacao ABC com estoque atual
beg['ValorEstoque'] = beg['onHand'] * beg['Price']
estoque_produto = beg.groupby('Brand')['ValorEstoque'].sum().reset_index()

capital_parado = receita[receita['Classe'] == 'C'].merge(estoque_produto, on='Brand', how='left')
capital_parado = capital_parado[capital_parado['ValorEstoque'] > 0].sort_values('ValorEstoque', ascending=False)

print(f'Produtos Classe C com estoque: {len(capital_parado):,}')
print(f'Capital parado em Classe C: R$ {capital_parado["ValorEstoque"].sum():,.2f}')
print(f'Isso representa {capital_parado["ValorEstoque"].sum() / beg["ValorEstoque"].sum() * 100:.1f}% do estoque total')

top_parado = capital_parado.head(15)
fig = px.bar(
    top_parado.sort_values('ValorEstoque'),
    x='ValorEstoque', y='Description',
    orientation='h',
    text='ValorEstoque',
    color='ValorEstoque',
    color_continuous_scale='Reds',
    title='Top 15 Produtos Classe C com Maior Capital Parado em Estoque',
    labels={'Description': 'Produto', 'ValorEstoque': 'Valor em Estoque (R$)'}
)
fig.update_traces(texttemplate='R$ %{text:,.0f}', textposition='outside')
fig.update_layout(height=520, coloraxis_showscale=False)
fig.show()

Produtos Classe C com estoque: 3,253
Capital parado em Classe C: R$ 8,969,370.35
Isso representa 13.2% do estoque total


## 6. Exportar Classificacao ABC

In [7]:
receita.to_csv('../data/processed/abc_classificacao.csv', index=False)
print(f'Exportado: abc_classificacao.csv ({len(receita):,} produtos classificados)')
print('\nDistribuicao final:')
print(receita['Classe'].value_counts().to_string())

Exportado: abc_classificacao.csv (6,396 produtos classificados)

Distribuicao final:
Classe
C    3408
B    1629
A    1359


## 7. Conclusoes da Curva ABC

- A Curva ABC confirma o principio de Pareto: poucos produtos concentram a maior parte da receita.
- **Classe A:** merece atencao maxima em gestao, reposicao prioritaria e nunca deve ter ruptura.
- **Classe C:** produtos de baixo giro que podem acumular capital parado. Candidatos a promocao ou descontinuidade.
- O cruzamento ABC x Estoque revela onde o dinheiro esta dormindo: produtos com pouca saida mas alto valor em prateleira.
- Para um pequeno empresario, focar nos produtos Classe A e revisar periodicamente a Classe C pode liberar capital significativo.
